# Experiment 3.1: Hessian-Adaptive Muon 2.0 -- Can a Cheap Hessian Sketch Close the Gap to Full Democratic SGD?

## The Central Claim This Experiment Tests

Experiment 2.11 established a striking result: equalizing gradient projections in the exact Hessian eigenbasis ("Full Democratic SGD") recovers approximately **150% of Muon's advantage** over SGD. The polar factor, which Muon uses, is a cheap approximation to this ideal equalization. This experiment asks: **can we get closer to that 150% with a computationally cheap Hessian sketch, rather than the full (and expensive) exact Hessian?**

## Theoretical Motivation: From Full Hessian to Lanczos Sketches

### The Democratic Gradient Principle

In a standard loss landscape $\mathcal{L}(\theta)$, the gradient $g = \nabla_\theta \mathcal{L}$ has unequal projections onto different curvature directions. If $\{v_i, \lambda_i\}$ are the Hessian eigenpairs, then:

$$g = \sum_i (g \cdot v_i) v_i$$

The projections $g \cdot v_i$ are typically **highly anisotropic**: large along high-curvature directions and small along low-curvature directions. This anisotropy means SGD takes large steps along directions that are already well-explored (high curvature = sharply curving = near a minimum in that direction) and small steps along directions that are under-explored (low curvature = flat = far from the minimum in that direction). This is the fundamental pathology of steepest descent.

**Democratic SGD** equalizes these projections: replace each $|g \cdot v_i|$ with $\langle |g \cdot v_i| \rangle$ (the mean magnitude), preserving signs. This forces equal exploration in all curvature directions.

### Why Muon Approximates This

The polar factor $UV^T$ of the gradient matrix $G = U \Sigma V^T$ sets all singular values to 1, which is an equalization -- but in the **SVD basis of the gradient itself**, not in the Hessian eigenbasis. For well-conditioned problems where $G$'s SVD aligns with the Hessian structure, these are similar. For ill-conditioned problems ($\kappa \gg 1$), the Hessian eigenbasis and the gradient's SVD basis diverge, and the polar factor's equalization is no longer optimal.

### The Lanczos Compromise

The full Hessian costs $O(n^2)$ gradient evaluations to compute (via finite differences) and $O(n^2)$ storage. For $n = 48$ parameters, this is tractable but does not scale. The **Lanczos algorithm** provides a middle ground: with $k$ Hessian-vector products (each costing 2 gradient evaluations), Lanczos produces the top-$k$ and bottom-$k$ Ritz vectors -- approximate eigenvectors of the Hessian corresponding to the largest and smallest eigenvalues.

The hypothesis is that equalizing in this **partial Hessian eigenbasis** (top + bottom Ritz vectors) while leaving the complementary subspace untouched captures most of the benefit of full equalization at a fraction of the cost. The extreme curvature directions (top and bottom) are precisely where the anisotropy is worst, so equalizing them may suffice.

### Muon + Hessian Rescaling: A Natural Gradient Hybrid

An alternative approach combines Muon's polar factor (for direction) with curvature-aware step sizing (for magnitude). For each layer's gradient $G = U \Sigma V^T$:
- The polar factor gives the update direction $UV^T$
- Each singular direction $u_i v_i^T$ is rescaled by $1/\sqrt{h_i}$ where $h_i$ is the curvature along that direction (estimated via a single Hessian-vector product)

This is a cheap approximation to the natural gradient $H^{-1} g$, restricted to the SVD basis of the current gradient.

## Experimental Setup

- **Architecture:** 3-layer deep linear network, all layers $4 \times 4$ (48 total parameters)
- **Target:** Ill-conditioned matrix with $\kappa = 1000$ (singular values $\{100, 10, 1, 0.1\}$)
- **Training:** 500 steps per run, Hessian/Lanczos recomputed every 50 steps
- **Seeds:** 5 independent random initializations
- **LR tuning:** Per-method sweep over 10 candidate learning rates; best LR used for final counted run

### The Six Optimizers

| Method | Description | Per-step cost | Equalization basis |
|--------|-------------|---------------|--------------------|
| **(a) SGD** | Vanilla steepest descent | 1 gradient | None |
| **(b) Muon** | Polar factor via $k=5$ Newton-Schulz | 1 gradient + 5 NS iters | Gradient SVD |
| **(c) Full Democratic** | Equalize in exact Hessian eigenbasis | 1 gradient + full Hessian/50 steps | Full Hessian |
| **(d) Muon 2.0 (Lanczos-10)** | Equalize in top-10 + bottom-10 Ritz vectors | 1 gradient + 10 Lanczos steps/50 steps | Partial Hessian (20 directions) |
| **(e) Muon 2.0 (Lanczos-5)** | Equalize in top-5 + bottom-5 Ritz vectors | 1 gradient + 5 Lanczos steps/50 steps | Partial Hessian (10 directions) |
| **(f) Muon + Hessian rescale** | Polar factor + $1/\sqrt{\text{curvature}}$ rescaling | 1 gradient + 4 HVPs/layer/step | Gradient SVD + curvature magnitudes |

### Tracked Metrics

- **Loss curves:** Per-step loss for all 6 optimizers across 500 steps
- **MatMul counts:** Total matrix multiplications as a proxy for computational cost
- **Final loss:** Loss at step 500, averaged over seeds
- **Recovery %:** What fraction of the Full Democratic advantage over SGD does each method capture?
- **Pareto score:** Recovery% / (relative cost), measuring efficiency

### Key Hypothesis Tests

- **T1:** Does Muon 2.0 (Lanczos-10) recover >80% of the Full Democratic advantage?
- **T2:** Does Muon 2.0 (Lanczos-10) beat standard Muon?
- **T3:** Does the cheaper variant (Lanczos-5) also beat standard Muon?
- **T4:** Does Muon + Hessian rescaling beat standard Muon?
- **T5:** Which method is Pareto-optimal (best recovery per unit of computational cost)?

In [ ]:
import numpy as np
import time

print("NumPy version:", np.__version__)
print("Environment ready.")

## Configuration: Network Architecture and Experimental Constants

### Why 4x4 Layers and 3 Layers Deep?

The dimension $n = 4$ is chosen so that the full Hessian ($48 \times 48$) is tractable via finite differences, enabling exact computation for the Full Democratic baseline. The 3-layer depth creates a nontrivial optimization landscape: the product $W_3 W_2 W_1$ has Hessian blocks that couple all three layers, and the loss surface has saddle points and flat directions that only appear in multi-layer settings.

### Why $\kappa = 1000$?

The target matrix has condition number $\kappa = 100/0.1 = 1000$. At this level of ill-conditioning:
- SGD's convergence is severely degraded ($O(\kappa^2) = O(10^6)$ effective condition number for convergence rate)
- The Hessian has eigenvalues spanning several orders of magnitude
- The gradient's SVD basis and the Hessian eigenbasis diverge significantly

This makes the problem hard enough that the choice of optimization direction matters enormously, but not so hard that SGD fails completely.

### Hessian Recomputation Interval

Recomputing the Hessian/Lanczos sketch every 50 steps is a compromise between cost and staleness. The Hessian changes as the parameters move, so a sketch computed at step $t$ becomes less accurate as a guide at step $t + \Delta t$. The 50-step interval was chosen to be:
- Short enough that the Hessian eigenvectors do not rotate by more than ~10-20 degrees between recomputations
- Long enough that the Lanczos cost is amortized over many steps (Lanczos-10 costs 20 gradient evaluations per recomputation, amortized to 0.4 gradients/step)

### Finite-Difference Epsilon

The Hessian-vector product $Hv$ is computed via central finite differences: $Hv \approx (g(\theta + \epsilon v) - g(\theta - \epsilon v)) / (2\epsilon)$ with $\epsilon = 10^{-5}$. This balances truncation error ($O(\epsilon^2)$) against floating-point roundoff ($O(\epsilon^{-1} \cdot \text{macheps})$). For 64-bit floats, $\epsilon = 10^{-5}$ gives approximately 5 digits of accuracy in the Hessian-vector product.

In [ ]:
# ── network / problem setup ──────────────────────────────────────────
DIM = 4
N_LAYERS = 3       # W3 @ W2 @ W1 -> T   (48 params total)
N_PARAMS = N_LAYERS * DIM * DIM
N_STEPS = 500
HESSIAN_RECOMPUTE_EVERY = 50
N_SEEDS = 5
EPS_FD = 1e-5       # finite-difference epsilon for Hessian-vector products

print("=" * 80)
print("EXPERIMENTAL CONFIGURATION")
print("=" * 80)
print(f"  Network: {N_LAYERS}-layer deep linear, {DIM}x{DIM} per layer")
print(f"  Total parameters: {N_PARAMS}")
print(f"  Hessian dimension: {N_PARAMS}x{N_PARAMS} = {N_PARAMS**2} entries")
print(f"  Training steps: {N_STEPS}")
print(f"  Hessian/Lanczos recompute interval: every {HESSIAN_RECOMPUTE_EVERY} steps")
print(f"  Number of recomputations per run: {N_STEPS // HESSIAN_RECOMPUTE_EVERY}")
print(f"  Seeds: {N_SEEDS}")
print(f"  Finite-difference epsilon: {EPS_FD}")
print(f"  Target condition number: 1000")
print()
print("  Singular values of target: [100.0, 10.0, 1.0, 0.1]")
print(f"  -> kappa = 100.0 / 0.1 = 1000")
print(f"  -> Dynamic range spans 3 orders of magnitude")
print(f"  -> This makes the Hessian eigenbasis and gradient SVD basis diverge,")
print(f"     which is precisely what the Lanczos methods aim to exploit.")

## MatMul Counter: Fair Cost Comparison

A crucial aspect of this experiment is **cost-aware comparison**. Different optimizers have different per-step computational costs:

- SGD: 1 gradient evaluation per step
- Muon: 1 gradient + 5 Newton-Schulz iterations (each involving 2 matrix multiplications)
- Full Democratic: 1 gradient + periodic full Hessian computation
- Muon 2.0: 1 gradient + periodic Lanczos iterations (each requiring 2 gradient evaluations)
- Muon + Hessian rescale: 1 gradient + multiple Hessian-vector products per step

We use a global matrix multiplication counter to track the total number of matmuls each optimizer consumes over the full training run. This enables Pareto-efficiency analysis: a method that achieves 90% of Full Democratic's advantage at 50% of Full Democratic's cost is more useful than one that achieves 95% at 200% of the cost.

The `counted_matmul` wrapper intercepts every matrix multiplication that goes through the optimizer's computational path, providing an architecture-independent proxy for floating-point cost.

In [ ]:
# ── matmul counter ───────────────────────────────────────────────────
_matmul_count = 0

def counted_matmul(A, B):
    """Matrix multiply with counting."""
    global _matmul_count
    _matmul_count += 1
    return A @ B

def reset_matmul_count():
    global _matmul_count
    _matmul_count = 0

def get_matmul_count():
    return _matmul_count

print("MatMul counter initialized.")
print("  Every matrix multiplication through counted_matmul increments the global counter.")
print("  This tracks computational cost across all optimizer variants.")

# Quick demo
reset_matmul_count()
A = np.random.randn(4, 4)
B = np.random.randn(4, 4)
_ = counted_matmul(A, B)
_ = counted_matmul(A, counted_matmul(B, A))
print(f"  Demo: 3 counted matmuls -> counter = {get_matmul_count()}")
reset_matmul_count()

## Core Primitives: Pack/Unpack, Forward Pass, Loss, and Gradient

### Parameter Vectorization

The 48-dimensional parameter vector $\theta \in \mathbb{R}^{48}$ is a flattened concatenation of the three $4 \times 4$ weight matrices: $\theta = [\text{vec}(W_1); \text{vec}(W_2); \text{vec}(W_3)]$. The `pack` and `unpack` functions handle this bijection.

### Forward Pass

The network output is the matrix product $Y = W_3 W_2 W_1$. This is a **deep linear network** -- despite having no nonlinearity, the optimization landscape has nontrivial structure because the loss is a **non-convex function** of the individual weight matrices $W_l$, even though it is convex in the product $\Pi = W_3 W_2 W_1$.

### Loss Function

The loss is the squared Frobenius norm of the residual:

$$\mathcal{L}(\theta) = \frac{1}{2} \| W_3 W_2 W_1 - T \|_F^2$$

where $T$ is the ill-conditioned target matrix. The factor $1/2$ simplifies the gradient.

### Gradient Computation

The gradient with respect to layer $k$ is:

$$\nabla_{W_k} \mathcal{L} = L_k^T \cdot (W_3 W_2 W_1 - T) \cdot R_k^T$$

where $L_k = \prod_{j > k} W_j$ (left context: all layers above $k$) and $R_k = \prod_{j < k} W_j$ (right context: all layers below $k$). This gradient depends on **all other layers**, which is the source of the inter-layer coupling that makes deep optimization difficult.

In [ ]:
# ── helpers ──────────────────────────────────────────────────────────

def pack(Ws):
    return np.concatenate([W.ravel() for W in Ws])

def unpack(theta):
    Ws = []
    idx = 0
    for _ in range(N_LAYERS):
        Ws.append(theta[idx:idx+DIM*DIM].reshape(DIM, DIM))
        idx += DIM*DIM
    return Ws

def forward(Ws, count=True):
    mm = counted_matmul if count else (lambda a, b: a @ b)
    out = Ws[0]
    for W in Ws[1:]:
        out = mm(W, out)
    return out

def loss_fn(theta, target, count=True):
    Ws = unpack(theta)
    diff = forward(Ws, count=count) - target
    return 0.5 * np.sum(diff**2)

def grad_fn(theta, target, count=True):
    """Compute gradient of loss w.r.t. theta (flat vector)."""
    Ws = unpack(theta)
    mm = counted_matmul if count else (lambda a, b: a @ b)
    prod = forward(Ws, count=count)
    R = prod - target

    grads = []
    for k in range(N_LAYERS):
        L = np.eye(DIM)
        for j in range(k+1, N_LAYERS):
            L = mm(Ws[j], L)
        Rp = np.eye(DIM)
        for j in range(0, k):
            Rp = mm(Ws[j], Rp)
        dWk = mm(L.T, mm(R, Rp.T))
        grads.append(dWk.ravel())
    return np.concatenate(grads)

def grad_matrices(theta, target, count=True):
    g = grad_fn(theta, target, count=count)
    mats = []
    for k in range(N_LAYERS):
        mats.append(g[k*DIM*DIM:(k+1)*DIM*DIM].reshape(DIM, DIM))
    return mats

print("Core functions defined: pack, unpack, forward, loss_fn, grad_fn, grad_matrices")
print()

# Sanity check
rng_test = np.random.RandomState(42)
theta_test = 0.3 * rng_test.randn(N_PARAMS)
U_test, _ = np.linalg.qr(rng_test.randn(DIM, DIM))
V_test, _ = np.linalg.qr(rng_test.randn(DIM, DIM))
target_test = U_test @ np.diag([100.0, 10.0, 1.0, 0.1]) @ V_test

reset_matmul_count()
L_test = loss_fn(theta_test, target_test)
g_test = grad_fn(theta_test, target_test)
gm_test = grad_matrices(theta_test, target_test, count=False)

print(f"Sanity check with random init (seed=42):")
print(f"  Loss: {L_test:.6f}")
print(f"  Gradient norm: {np.linalg.norm(g_test):.6f}")
print(f"  Gradient matrices norms: {[f'{np.linalg.norm(gm, "fro"):.4f}' for gm in gm_test]}")
print(f"  MatMuls for one loss + gradient eval: {get_matmul_count()}")
print()
print(f"  Target matrix condition number: {np.linalg.cond(target_test):.1f}")
print(f"  Target singular values: {np.linalg.svd(target_test, compute_uv=False).round(2)}")
reset_matmul_count()

## Hessian Computation: The Oracle Baseline

### Full Hessian via Finite Differences

The full $48 \times 48$ Hessian is computed column-by-column using central finite differences:

$$H_{:,i} = \frac{g(\theta + \epsilon e_i) - g(\theta - \epsilon e_i)}{2\epsilon}$$

This requires $2n = 96$ gradient evaluations. The result is symmetrized via $H \leftarrow \frac{1}{2}(H + H^T)$ to correct for numerical asymmetry.

**This is prohibitively expensive for large-scale problems** -- the cost scales as $O(n^2)$ gradient evaluations. It serves here only as the gold-standard oracle to define the upper bound on democratic equalization.

### Hessian-Vector Product via Finite Differences

For the Lanczos and Hessian-rescaling methods, we need only Hessian-vector products $Hv$, not the full Hessian. Each HVP costs only 2 gradient evaluations regardless of dimensionality:

$$Hv = \frac{g(\theta + \epsilon v) - g(\theta - \epsilon v)}{2\epsilon}$$

This is the key to making the Lanczos approach scalable: $k$ Lanczos steps require $2k$ gradient evaluations, not $2n$.

### Cost Comparison

| Operation | Cost (gradient evaluations) | For $n=48$ |
|-----------|---------------------------|------------|
| Full Hessian | $2n$ | 96 |
| Lanczos-10 | $2 \times 10 = 20$ | 20 |
| Lanczos-5 | $2 \times 5 = 10$ | 10 |
| Per-layer HVP (Hessian rescale) | $2 \times n_{\text{layer}} \times L$ | 24 |

The Lanczos methods are 5-10x cheaper than the full Hessian, making them viable for periodic recomputation.

In [ ]:
# ── Hessian (exact, for Full Democratic and verification) ────────────

def hessian_fn(theta, target):
    """Full Hessian via finite differences (no matmul counting -- overhead)."""
    n = len(theta)
    H = np.zeros((n, n))
    for i in range(n):
        theta_p = theta.copy()
        theta_m = theta.copy()
        theta_p[i] += EPS_FD
        theta_m[i] -= EPS_FD
        g_p = grad_fn(theta_p, target, count=False)
        g_m = grad_fn(theta_m, target, count=False)
        H[:, i] = (g_p - g_m) / (2 * EPS_FD)
    H = 0.5 * (H + H.T)
    return H

# ── Hessian-vector product via finite differences ────────────────────

def hvp(theta, target, v):
    """Hessian-vector product Hv via central finite differences.
    Requires 2 gradient evaluations. These ARE counted for matmul cost."""
    theta_p = theta + EPS_FD * v
    theta_m = theta - EPS_FD * v
    g_p = grad_fn(theta_p, target, count=True)
    g_m = grad_fn(theta_m, target, count=True)
    return (g_p - g_m) / (2 * EPS_FD)

print("Hessian and HVP functions defined.")
print()

# Demonstrate Hessian computation
print("Computing full Hessian at test point for verification...")
H_test = hessian_fn(theta_test, target_test)
evals_test = np.linalg.eigvalsh(H_test)
print(f"  Hessian shape: {H_test.shape}")
print(f"  Hessian eigenvalue range: [{evals_test[0]:.4f}, {evals_test[-1]:.4f}]")
print(f"  Hessian spectral condition number: {evals_test[-1] / (abs(evals_test[0]) + 1e-12):.1f}")
print(f"  Number of positive eigenvalues: {np.sum(evals_test > 0)}")
print(f"  Number of negative eigenvalues: {np.sum(evals_test < 0)}")
print(f"  Number of near-zero eigenvalues (|lambda| < 0.01): {np.sum(np.abs(evals_test) < 0.01)}")
print()

# Verify HVP against full Hessian
v_test = np.random.randn(N_PARAMS)
v_test /= np.linalg.norm(v_test)
reset_matmul_count()
hvp_result = hvp(theta_test, target_test, v_test)
hvp_exact = H_test @ v_test
hvp_error = np.linalg.norm(hvp_result - hvp_exact) / np.linalg.norm(hvp_exact)
print(f"  HVP verification (random direction):")
print(f"    Relative error |Hv_FD - Hv_exact| / |Hv_exact| = {hvp_error:.2e}")
print(f"    MatMuls for one HVP: {get_matmul_count()}")
print(f"    (This confirms the finite-difference HVP matches the exact Hessian.)")
reset_matmul_count()

## Lanczos Tridiagonalization: Cheap Hessian Spectral Sketch

### The Lanczos Algorithm

The Lanczos algorithm is an iterative method for approximating the extreme eigenvalues and eigenvectors of a symmetric matrix, using only matrix-vector products. It builds an orthonormal basis $\{q_1, q_2, \ldots, q_k\}$ for the Krylov subspace $\mathcal{K}_k(H, q_1) = \text{span}\{q_1, Hq_1, H^2 q_1, \ldots, H^{k-1} q_1\}$, and represents $H$ restricted to this subspace as a tridiagonal matrix $T_k$.

The eigenvalues of $T_k$ (called **Ritz values**) approximate the extreme eigenvalues of $H$, and the corresponding eigenvectors mapped back to the full space (called **Ritz vectors**) approximate the extreme eigenvectors.

### Algorithm

1. Start with random unit vector $q_1$
2. For $j = 1, \ldots, k$:
   - Compute $v = H q_j$ (one Hessian-vector product)
   - $\alpha_j = q_j^T v$ (diagonal element of $T$)
   - Orthogonalize: $v \leftarrow v - \alpha_j q_j - \beta_j q_{j-1}$ (three-term recurrence)
   - Full reorthogonalization against all previous $q$'s (for numerical stability)
   - $\beta_{j+1} = \|v\|$, $q_{j+1} = v / \beta_{j+1}$
3. Eigendecompose the $k \times k$ tridiagonal $T_k$
4. Map Ritz vectors back: $V_{\text{Ritz}} = Q \cdot V_{\text{small}}$

### Why Full Reorthogonalization?

In exact arithmetic, the Lanczos vectors are automatically orthogonal. In finite-precision arithmetic, orthogonality is lost due to roundoff, causing "ghost eigenvalues" (spurious duplicates of converged eigenvalues). Full reorthogonalization at each step prevents this at the cost of $O(k^2 n)$ additional work -- negligible compared to the $k$ Hessian-vector products for the problem sizes we consider.

### What Lanczos Captures

With $k$ steps, Lanczos typically captures:
- The **largest** $\sim k/2$ eigenvalues accurately (these converge fastest)
- The **smallest** $\sim k/2$ eigenvalues reasonably well
- Poor approximation for eigenvalues in the interior of the spectrum

For the democratic equalization goal, this is ideal: the extreme eigenvalues correspond to the directions with the most severe anisotropy (too much curvature at the top, too little at the bottom). Equalizing in these directions captures the bulk of the benefit.

In [ ]:
# ── Lanczos tridiagonalization ───────────────────────────────────────

def lanczos(theta, target, k):
    """Run k steps of Lanczos to get an approximate eigendecomposition.
    Returns (eigenvalues, eigenvectors_in_full_space) of the top-k and bottom-k
    Ritz vectors.

    Cost: k Hessian-vector products = 2k gradient evaluations.
    """
    n = len(theta)
    # Initialize with a random unit vector (seeded for reproducibility)
    rng = np.random.RandomState(int(np.abs(theta[:4]).sum() * 1000) % (2**31))
    q = rng.randn(n)
    q = q / np.linalg.norm(q)

    Q = np.zeros((n, k))
    alpha = np.zeros(k)
    beta = np.zeros(k)

    Q[:, 0] = q
    for j in range(k):
        v = hvp(theta, target, Q[:, j])
        alpha[j] = Q[:, j] @ v
        if j == 0:
            v = v - alpha[j] * Q[:, j]
        else:
            v = v - alpha[j] * Q[:, j] - beta[j] * Q[:, j-1]

        # Full reorthogonalization for numerical stability
        for jj in range(j+1):
            v = v - (Q[:, jj] @ v) * Q[:, jj]

        beta_next = np.linalg.norm(v)
        if beta_next < 1e-12:
            # Krylov subspace exhausted, pad with zeros
            if j + 1 < k:
                beta[j+1] = 0.0
            break
        if j + 1 < k:
            beta[j+1] = beta_next
            Q[:, j+1] = v / beta_next

    # Build tridiagonal matrix
    T = np.diag(alpha)
    for j in range(k-1):
        T[j, j+1] = beta[j+1]
        T[j+1, j] = beta[j+1]

    # Eigendecompose the tridiagonal
    ritz_vals, ritz_vecs_small = np.linalg.eigh(T)

    # Map back to full space
    ritz_vecs = Q @ ritz_vecs_small

    # Normalize (should already be near-unit, but ensure)
    for i in range(ritz_vecs.shape[1]):
        norm = np.linalg.norm(ritz_vecs[:, i])
        if norm > 1e-12:
            ritz_vecs[:, i] /= norm

    return ritz_vals, ritz_vecs

print("Lanczos tridiagonalization function defined.")
print()

# Demonstrate Lanczos accuracy vs exact Hessian
print("Verifying Lanczos accuracy against exact Hessian eigendecomposition...")
reset_matmul_count()
ritz_vals_10, ritz_vecs_10 = lanczos(theta_test, target_test, 10)
lanczos10_cost = get_matmul_count()
reset_matmul_count()
ritz_vals_5, ritz_vecs_5 = lanczos(theta_test, target_test, 5)
lanczos5_cost = get_matmul_count()
reset_matmul_count()

print(f"\n  Exact Hessian eigenvalues (all {N_PARAMS}):")
print(f"    Smallest 5: {evals_test[:5].round(4)}")
print(f"    Largest 5:  {evals_test[-5:].round(4)}")
print()
print(f"  Lanczos-10 Ritz values ({len(ritz_vals_10)} values):")
print(f"    {ritz_vals_10.round(4)}")
print(f"    MatMul cost: {lanczos10_cost}")
print()
print(f"  Lanczos-5 Ritz values ({len(ritz_vals_5)} values):")
print(f"    {ritz_vals_5.round(4)}")
print(f"    MatMul cost: {lanczos5_cost}")
print()

# Check alignment of Ritz vectors with true eigenvectors
_, true_evecs = np.linalg.eigh(H_test)
print(f"  Ritz vector alignment with true eigenvectors (top Ritz vs top true):")
for i in range(min(3, len(ritz_vals_10))):
    alignment = abs(ritz_vecs_10[:, -(i+1)] @ true_evecs[:, -(i+1)])
    print(f"    Ritz #{i+1} <-> True #{i+1}: |cos(angle)| = {alignment:.4f}")
print(f"    (1.0 = perfect alignment, 0.0 = orthogonal)")

## Optimizer Directions: Six Strategies Compared

This section defines the core update direction for each of the six optimizers. Each one represents a different point on the spectrum from "use the raw gradient" (SGD) to "equalize perfectly in the Hessian eigenbasis" (Full Democratic).

### (b) Muon: Polar Factor via Newton-Schulz

For each layer's $4 \times 4$ gradient matrix $G_k$, compute the polar factor $U_k V_k^T$ via 5 Newton-Schulz iterations. The update direction for layer $k$ is this polar factor. All singular values of $U_k V_k^T$ are exactly 1, so the gradient is equalized **in the SVD basis of the gradient itself**.

### (c) Full Democratic: Equalize in Hessian Eigenbasis

Project the full gradient $g \in \mathbb{R}^{48}$ onto the Hessian eigenvectors $\{v_i\}_{i=1}^{48}$. Replace each projection magnitude $|g \cdot v_i|$ with the mean $\bar{m} = \frac{1}{48} \sum_i |g \cdot v_i|$, preserving signs. The reconstructed direction has equal magnitude in all curvature directions.

### (d,e) Muon 2.0 (Lanczos): Partial Equalization

Project the gradient onto the $k$ Ritz vectors from Lanczos. Equalize magnitudes within this subspace (same as Full Democratic, but restricted to $k$ directions). Add back the unmodified complement (the gradient's component orthogonal to the Ritz subspace). This creates a "partially equalized" direction: democratic in the extreme curvature directions, raw-SGD in the rest.

### (f) Muon + Hessian Rescaling

For each layer's gradient $G_k = U_k \Sigma_k V_k^T$:
1. Estimate the curvature $h_i$ along each singular direction $u_{k,i} v_{k,i}^T$ via one Hessian-vector product
2. Construct the rescaled direction: $\sum_i \frac{1}{\sqrt{h_i}} u_{k,i} v_{k,i}^T$

This is a hybrid between Muon (which uses $UV^T$ with uniform weighting) and Newton's method (which uses $H^{-1}g$ with full inverse-curvature weighting). The $1/\sqrt{h_i}$ weighting is a compromise: it corrects for curvature without fully inverting it, which is more stable.

In [ ]:
# ── Optimizer directions ─────────────────────────────────────────────

def polar_factor_ns(M, k=5):
    """Polar factor via Newton-Schulz iterations (k iterations)."""
    # Normalize
    a = np.linalg.norm(M, 'fro')
    if a < 1e-12:
        return M
    X = M / a
    for _ in range(k):
        X = 1.5 * X - 0.5 * counted_matmul(X, counted_matmul(X.T, X))
    return X

def polar_factor_svd(M):
    """Polar factor via SVD (exact, for reference)."""
    U, S, Vt = np.linalg.svd(M, full_matrices=True)
    return counted_matmul(U, Vt)

def muon_direction(theta, target):
    """Standard Muon: polar factor of each layer's gradient matrix."""
    gmats = grad_matrices(theta, target, count=True)
    polars = []
    for gm in gmats:
        polars.append(polar_factor_ns(gm, k=5).ravel())
    return np.concatenate(polars)

def democratic_direction(grad_vec, eigvecs):
    """Full Democratic SGD: equalize projections in Hessian eigenbasis."""
    projs = eigvecs.T @ grad_vec
    signs = np.sign(projs)
    magnitudes = np.abs(projs)
    mean_mag = np.mean(magnitudes)
    eq_projs = signs * mean_mag
    return eigvecs @ eq_projs

def muon2_lanczos_direction(grad_vec, ritz_vecs, ritz_vals):
    """Muon 2.0 (Lanczos): equalize gradient in the Lanczos subspace,
    keep SGD direction in the complement.

    ritz_vecs: (n, 2m) matrix of m top + m bottom Ritz vectors.
    """
    n = len(grad_vec)
    m = ritz_vecs.shape[1]

    # Project gradient onto Lanczos subspace
    projs = ritz_vecs.T @ grad_vec       # shape (m,)

    # Equalize magnitudes in the Lanczos subspace
    signs = np.sign(projs)
    magnitudes = np.abs(projs)
    mean_mag = np.mean(magnitudes) if np.max(magnitudes) > 1e-30 else 0.0
    eq_projs = signs * mean_mag

    # Reconstructed equalized component in Lanczos subspace
    equalized_part = ritz_vecs @ eq_projs

    # Complement: gradient minus its projection onto Lanczos subspace
    grad_in_subspace = ritz_vecs @ projs
    complement = grad_vec - grad_in_subspace

    # Full direction: equalized Lanczos + raw SGD complement
    direction = equalized_part + complement
    return direction

def muon_hessian_rescale_direction(theta, target):
    """Muon + Hessian rescaling: polar factor step rescaled by 1/sqrt(curvature)
    in each singular direction.

    For each layer gradient G = U S V^T:
      - Polar factor gives U V^T
      - Estimate curvature in each singular direction via a cheap Hv product
      - Rescale: sum_i (1/sqrt(|h_i| + eps)) * u_i v_i^T
    """
    gmats = grad_matrices(theta, target, count=True)
    directions = []

    for layer_idx, gm in enumerate(gmats):
        U, S, Vt = np.linalg.svd(gm, full_matrices=True)
        _mc = 3  # SVD counts as ~3 matmuls

        # Estimate curvature along each singular direction
        curvatures = np.zeros(DIM)
        offset = layer_idx * DIM * DIM
        for i in range(DIM):
            # Direction in parameter space corresponding to u_i v_i^T
            direction_mat = np.outer(U[:, i], Vt[i, :])
            v_full = np.zeros(N_PARAMS)
            v_full[offset:offset+DIM*DIM] = direction_mat.ravel()

            # Hessian-vector product
            Hv = hvp(theta, target, v_full)
            curvatures[i] = np.abs(v_full @ Hv) + 1e-8

        # Rescale: polar factor with 1/sqrt(curvature) weighting
        rescaled = np.zeros((DIM, DIM))
        for i in range(DIM):
            weight = 1.0 / np.sqrt(curvatures[i])
            rescaled += weight * np.outer(U[:, i], Vt[i, :])

        directions.append(rescaled.ravel())

    return np.concatenate(directions)

print("All six optimizer directions defined:")
print("  (a) SGD: raw gradient")
print("  (b) Muon: polar factor via Newton-Schulz (k=5)")
print("  (c) Full Democratic SGD: equalize in exact Hessian eigenbasis")
print("  (d) Muon 2.0 (Lanczos-10): equalize in top-10 Ritz subspace")
print("  (e) Muon 2.0 (Lanczos-5): equalize in top-5 Ritz subspace")
print("  (f) Muon + Hessian rescale: polar factor with curvature-weighted magnitudes")
print()

# Demonstrate directions at test point
reset_matmul_count()
g_test_vec = grad_fn(theta_test, target_test, count=False)
d_muon = muon_direction(theta_test, target_test)

H_test_full = hessian_fn(theta_test, target_test)
_, H_evecs = np.linalg.eigh(H_test_full)
d_dem = democratic_direction(g_test_vec, H_evecs)

rv10, re10 = lanczos(theta_test, target_test, 10)
d_l10 = muon2_lanczos_direction(g_test_vec, re10, rv10)

print(f"Direction comparison at test point:")
print(f"  SGD (gradient) norm:        {np.linalg.norm(g_test_vec):.4f}")
print(f"  Muon direction norm:         {np.linalg.norm(d_muon):.4f}")
print(f"  Democratic direction norm:    {np.linalg.norm(d_dem):.4f}")
print(f"  Lanczos-10 direction norm:   {np.linalg.norm(d_l10):.4f}")
print()
cos_muon_dem = np.dot(d_muon, d_dem) / (np.linalg.norm(d_muon) * np.linalg.norm(d_dem) + 1e-12)
cos_l10_dem = np.dot(d_l10, d_dem) / (np.linalg.norm(d_l10) * np.linalg.norm(d_dem) + 1e-12)
cos_sgd_dem = np.dot(g_test_vec, d_dem) / (np.linalg.norm(g_test_vec) * np.linalg.norm(d_dem) + 1e-12)
print(f"  Cosine similarity with Full Democratic direction:")
print(f"    SGD vs Democratic:         {cos_sgd_dem:.4f}")
print(f"    Muon vs Democratic:        {cos_muon_dem:.4f}")
print(f"    Lanczos-10 vs Democratic:  {cos_l10_dem:.4f}")
print(f"    (Higher = closer to the ideal democratic direction)")
reset_matmul_count()

## Learning Rate Sweep and Non-Counting Utility Functions

### Fair Comparison via Per-Method LR Tuning

Each optimizer gets its own optimal learning rate, determined by a grid search over 10 candidates spanning $[10^{-4}, 0.1]$. This is critical for a fair comparison: different update directions have different effective magnitudes, and comparing optimizers at a single learning rate would confound direction quality with magnitude scaling.

The LR sweep uses **non-counting** versions of all functions (no matmul counter increments) because the LR search is a tuning cost that should not penalize the optimizer in the final Pareto analysis. Only the final counted run contributes to the matmul cost.

### Non-Counting Variants

The `_nocost` variants of the Muon direction, Lanczos, and Hessian-rescale functions are identical to their counted counterparts except that matrix multiplications go through `a @ b` instead of `counted_matmul(a, b)`. This is purely an accounting distinction -- the numerical results are identical.

### Divergence Guard

If the loss exceeds $10^8$ or becomes NaN at any step, the run is terminated and a sentinel value of $10^8$ is returned. This prevents numerical instability from corrupting the LR sweep results.

In [ ]:
# ── LR sweep helper ─────────────────────────────────────────────────

def run_single(method, lr, theta0, target, lanczos_cache=None):
    """Run optimizer for N_STEPS, return final loss. No matmul counting."""
    theta = theta0.copy()
    H_eigvecs = None
    ritz_vecs_10 = None
    ritz_vals_10 = None
    ritz_vecs_5 = None
    ritz_vals_5 = None

    for step in range(N_STEPS):
        g = grad_fn(theta, target, count=False)

        if step % HESSIAN_RECOMPUTE_EVERY == 0:
            if method in ('FullDemocratic',):
                H = hessian_fn(theta, target)
                _, H_eigvecs = np.linalg.eigh(H)

            if method == 'Muon2_L10':
                ritz_vals_10, ritz_vecs_10 = lanczos_nocost(theta, target, 10)
            elif method == 'Muon2_L5':
                ritz_vals_5, ritz_vecs_5 = lanczos_nocost(theta, target, 5)

        if method == 'SGD':
            theta -= lr * g
        elif method == 'Muon':
            d = muon_direction_nocost(theta, target)
            theta -= lr * d
        elif method == 'FullDemocratic':
            g_dem = democratic_direction(g, H_eigvecs)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(g_dem)
            if dn > 1e-12:
                g_dem = g_dem * (gn / dn)
            theta -= lr * g_dem
        elif method == 'Muon2_L10':
            d = muon2_lanczos_direction(g, ritz_vecs_10, ritz_vals_10)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(d)
            if dn > 1e-12:
                d = d * (gn / dn)
            theta -= lr * d
        elif method == 'Muon2_L5':
            d = muon2_lanczos_direction(g, ritz_vecs_5, ritz_vals_5)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(d)
            if dn > 1e-12:
                d = d * (gn / dn)
            theta -= lr * d
        elif method == 'Muon+Hrescale':
            d = muon_hrescale_nocost(theta, target)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(d)
            if dn > 1e-12:
                d = d * (gn / dn)
            theta -= lr * d

        L = loss_fn(theta, target, count=False)
        if np.isnan(L) or L > 1e8:
            return 1e8
    return loss_fn(theta, target, count=False)

# Non-counting versions for LR sweep
def muon_direction_nocost(theta, target):
    gmats = grad_matrices(theta, target, count=False)
    polars = []
    for gm in gmats:
        U, S, Vt = np.linalg.svd(gm, full_matrices=True)
        X = gm / (np.linalg.norm(gm, 'fro') + 1e-12)
        for _ in range(5):
            X = 1.5 * X - 0.5 * X @ (X.T @ X)
        polars.append(X.ravel())
    return np.concatenate(polars)

def lanczos_nocost(theta, target, k):
    """Lanczos without matmul counting."""
    n = len(theta)
    rng = np.random.RandomState(int(np.abs(theta[:4]).sum() * 1000) % (2**31))
    q = rng.randn(n)
    q = q / np.linalg.norm(q)

    Q = np.zeros((n, k))
    alpha = np.zeros(k)
    beta = np.zeros(k)
    Q[:, 0] = q

    for j in range(k):
        # HVP without counting
        v_dir = Q[:, j]
        g_p = grad_fn(theta + EPS_FD * v_dir, target, count=False)
        g_m = grad_fn(theta - EPS_FD * v_dir, target, count=False)
        v = (g_p - g_m) / (2 * EPS_FD)

        alpha[j] = Q[:, j] @ v
        if j == 0:
            v = v - alpha[j] * Q[:, j]
        else:
            v = v - alpha[j] * Q[:, j] - beta[j] * Q[:, j-1]
        for jj in range(j+1):
            v = v - (Q[:, jj] @ v) * Q[:, jj]
        beta_next = np.linalg.norm(v)
        if beta_next < 1e-12:
            break
        if j + 1 < k:
            beta[j+1] = beta_next
            Q[:, j+1] = v / beta_next

    T = np.diag(alpha)
    for j in range(k-1):
        T[j, j+1] = beta[j+1]
        T[j+1, j] = beta[j+1]
    ritz_vals, ritz_vecs_small = np.linalg.eigh(T)
    ritz_vecs = Q @ ritz_vecs_small
    for i in range(ritz_vecs.shape[1]):
        norm = np.linalg.norm(ritz_vecs[:, i])
        if norm > 1e-12:
            ritz_vecs[:, i] /= norm
    return ritz_vals, ritz_vecs

def muon_hrescale_nocost(theta, target):
    gmats = grad_matrices(theta, target, count=False)
    directions = []
    for layer_idx, gm in enumerate(gmats):
        U, S, Vt = np.linalg.svd(gm, full_matrices=True)
        curvatures = np.zeros(DIM)
        offset = layer_idx * DIM * DIM
        for i in range(DIM):
            direction_mat = np.outer(U[:, i], Vt[i, :])
            v_full = np.zeros(N_PARAMS)
            v_full[offset:offset+DIM*DIM] = direction_mat.ravel()
            g_p = grad_fn(theta + EPS_FD * v_full, target, count=False)
            g_m = grad_fn(theta - EPS_FD * v_full, target, count=False)
            Hv = (g_p - g_m) / (2 * EPS_FD)
            curvatures[i] = np.abs(v_full @ Hv) + 1e-8
        rescaled = np.zeros((DIM, DIM))
        for i in range(DIM):
            weight = 1.0 / np.sqrt(curvatures[i])
            rescaled += weight * np.outer(U[:, i], Vt[i, :])
        directions.append(rescaled.ravel())
    return np.concatenate(directions)

print("LR sweep and non-counting utility functions defined.")
print(f"LR candidates for sweep: [0.0001, 0.0003, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]")
print(f"Each method will be tested at all 10 LRs; best is selected for the counted run.")

## Full Counted Run: Training with MatMul Accounting

The `run_full_counted` function executes the complete training loop with all matmul operations going through the counter. This is the definitive run from which all final metrics are derived:

- **Loss curve:** Recorded at every step for detailed trajectory analysis
- **Final loss:** The loss at step 500, the primary performance metric
- **MatMul count:** Total matmuls consumed, the primary cost metric

### Periodic Hessian/Lanczos Recomputation

For methods that use Hessian information (Full Democratic, Muon 2.0 variants), the spectral sketch is recomputed every 50 steps. Between recomputations, the cached eigenvectors/Ritz vectors are reused. This creates a trade-off:

- **Fresh sketch (small interval):** More accurate curvature information, but higher cost
- **Stale sketch (large interval):** Cheaper, but the Hessian eigenvectors rotate during training, making the cached vectors increasingly inaccurate

The 50-step interval is a reasonable compromise for this 500-step experiment, yielding 10 recomputations per run.

### Norm Matching

For methods that produce a different-norm direction than the gradient (Democratic, Lanczos, Hessian rescale), the direction is rescaled to have the same norm as the original gradient. This ensures that the learning rate has consistent meaning across methods -- the comparison is purely about **direction quality**, not magnitude.

In [ ]:
# ── Full run with counting ───────────────────────────────────────────

def run_full_counted(method, lr, theta0, target):
    """Full run with matmul counting and loss curve."""
    reset_matmul_count()
    theta = theta0.copy()
    losses = []
    H_eigvecs = None
    ritz_vecs = None
    ritz_vals = None

    for step in range(N_STEPS):
        L = loss_fn(theta, target, count=True)
        losses.append(L)
        g = grad_fn(theta, target, count=True)

        if step % HESSIAN_RECOMPUTE_EVERY == 0:
            if method == 'FullDemocratic':
                H = hessian_fn(theta, target)
                _, H_eigvecs = np.linalg.eigh(H)

            if method == 'Muon2_L10':
                ritz_vals, ritz_vecs = lanczos(theta, target, 10)
            elif method == 'Muon2_L5':
                ritz_vals, ritz_vecs = lanczos(theta, target, 5)

        if method == 'SGD':
            theta -= lr * g
        elif method == 'Muon':
            d = muon_direction(theta, target)
            theta -= lr * d
        elif method == 'FullDemocratic':
            g_dem = democratic_direction(g, H_eigvecs)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(g_dem)
            if dn > 1e-12:
                g_dem = g_dem * (gn / dn)
            theta -= lr * g_dem
        elif method == 'Muon2_L10':
            d = muon2_lanczos_direction(g, ritz_vecs, ritz_vals)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(d)
            if dn > 1e-12:
                d = d * (gn / dn)
            theta -= lr * d
        elif method == 'Muon2_L5':
            d = muon2_lanczos_direction(g, ritz_vecs, ritz_vals)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(d)
            if dn > 1e-12:
                d = d * (gn / dn)
            theta -= lr * d
        elif method == 'Muon+Hrescale':
            d = muon_hessian_rescale_direction(theta, target)
            gn = np.linalg.norm(g)
            dn = np.linalg.norm(d)
            if dn > 1e-12:
                d = d * (gn / dn)
            theta -= lr * d

        if np.isnan(losses[-1]) or losses[-1] > 1e8:
            return np.array(losses), 1e8, get_matmul_count()

    final = loss_fn(theta, target, count=True)
    return np.array(losses), final, get_matmul_count()

print("Full counted run function defined.")
print("  Records: loss curve, final loss, total matmul count.")
print(f"  Hessian/Lanczos recomputation occurs at steps: {list(range(0, N_STEPS, HESSIAN_RECOMPUTE_EVERY))}")

## Main Experiment Loop: LR Sweep + Counted Run Over 5 Seeds

For each of 5 random seeds, we:

1. **Generate a fresh target and initialization:** The target matrix has fixed singular values $\{100, 10, 1, 0.1\}$ but random left and right singular vectors. The initial parameter vector is $\theta_0 \sim 0.3 \cdot \mathcal{N}(0, I_{48})$.

2. **Sweep learning rates for all 6 methods:** Each method is run at 10 LR candidates using the non-counting variant. The LR that achieves the lowest final loss is selected.

3. **Execute the full counted run at the best LR:** All matmuls are tracked, and the complete loss curve is recorded.

### Why Multiple Seeds?

The random singular vectors of the target create different curvature landscapes at each seed. A method that only works for one particular alignment of the target's SVD with the coordinate axes is not genuinely better -- it just got lucky. Averaging over 5 seeds provides a basic measure of robustness.

### Why $\kappa = 1000$ is Fixed

The condition number is held fixed because the experiment's purpose is to compare optimization *methods*, not to study how methods scale with conditioning (that is Experiment 2.11's domain). Fixing $\kappa$ ensures all methods face the same difficulty level.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════

METHOD_NAMES = ['SGD', 'Muon', 'FullDemocratic', 'Muon2_L10', 'Muon2_L5', 'Muon+Hrescale']
DISPLAY_NAMES = {
    'SGD': 'SGD (baseline)',
    'Muon': 'Muon (polar, k=5 NS)',
    'FullDemocratic': 'Full Democratic SGD',
    'Muon2_L10': 'Muon 2.0 (Lanczos-10)',
    'Muon2_L5': 'Muon 2.0 (Lanczos-5)',
    'Muon+Hrescale': 'Muon + Hessian rescale',
}

print("=" * 90)
print("Experiment 3.1: Hessian-Adaptive Muon 2.0")
print("=" * 90)
print(f"Network: {N_LAYERS}-layer deep linear {DIM}x{DIM}  ({N_PARAMS} params)")
print(f"Steps: {N_STEPS},  Hessian/Lanczos recompute every {HESSIAN_RECOMPUTE_EVERY} steps")
print(f"Seeds: {N_SEEDS}")
print(f"Target condition number: 1000")
print()

# LR candidates -- broader range for the new methods
lr_candidates = [0.0001, 0.0003, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]

all_results = {m: {'final_losses': [], 'matmuls': [], 'loss_curves': []} for m in METHOD_NAMES}

t_start = time.time()

for seed_idx in range(N_SEEDS):
    seed = 42 + seed_idx * 7
    rng_init = np.random.RandomState(seed)

    # Ill-conditioned target (kappa = 1000)
    U_t, _ = np.linalg.qr(rng_init.randn(DIM, DIM))
    V_t, _ = np.linalg.qr(rng_init.randn(DIM, DIM))
    sigma_t = np.array([100.0, 10.0, 1.0, 0.1])  # kappa = 100/0.1 = 1000
    target = U_t @ np.diag(sigma_t) @ V_t
    theta0 = 0.3 * rng_init.randn(N_PARAMS)

    kappa = np.linalg.cond(target)
    print(f"--- Seed {seed} (target kappa={kappa:.0f}) ---")

    # LR sweep for each method
    best_lrs = {}
    for method in METHOD_NAMES:
        best_loss = 1e20
        best_lr = 0.001
        for lr in lr_candidates:
            fl = run_single(method, lr, theta0, target)
            if fl < best_loss:
                best_loss = fl
                best_lr = lr
        best_lrs[method] = best_lr

    lr_str = ", ".join([f"{DISPLAY_NAMES[m].split('(')[0].strip()[:8]}={best_lrs[m]}" for m in METHOD_NAMES])
    print(f"  Best LRs: {lr_str}")

    # Full counted run
    for method in METHOD_NAMES:
        losses, final, matmuls = run_full_counted(method, best_lrs[method], theta0, target)
        all_results[method]['final_losses'].append(final)
        all_results[method]['matmuls'].append(matmuls)
        all_results[method]['loss_curves'].append(losses)

    # Quick per-seed summary
    finals = {m: all_results[m]['final_losses'][-1] for m in METHOD_NAMES}
    print(f"  Finals: " + "  ".join([f"{m[:6]}={finals[m]:.4f}" for m in METHOD_NAMES]))
    print()

elapsed = time.time() - t_start
print(f"\nTotal runtime: {elapsed:.1f}s\n")

## Aggregate Statistics: Cross-Seed Means and Standard Deviations

### Recovery Percentage

The recovery metric normalizes each method's performance against the best (Full Democratic) and worst (SGD) baselines:

$$\text{Recovery}(m) = \frac{\mathcal{L}_{\text{SGD}} - \mathcal{L}_m}{\mathcal{L}_{\text{SGD}} - \mathcal{L}_{\text{FullDemocratic}}} \times 100\%$$

- $0\%$: Method performs identically to SGD (no benefit from curvature-aware direction)
- $100\%$: Method matches Full Democratic SGD (the oracle upper bound)
- $> 100\%$: Method somehow exceeds Full Democratic (unexpected, would indicate the LR sweep found a better regime)

This is computed **per-seed** and then averaged, which is more robust than computing recovery on the mean losses (the latter can be distorted by outlier seeds).

### Pareto Score

$$\text{Pareto Score}(m) = \frac{\text{Recovery}(m)}{\text{MatMuls}(m) / \text{MatMuls}(\text{SGD})}$$

This measures efficiency: recovery per unit of computational cost. A method with 80% recovery at 2x the cost of SGD has a Pareto score of 40. A method with 50% recovery at 1.1x cost has a score of ~45. The Pareto score identifies methods that are not just accurate but also computationally efficient -- the most actionable result for practical optimizer design.

In [ ]:
# ── Aggregate results ────────────────────────────────────────────────
print("=" * 90)
print("AGGREGATE RESULTS")
print("=" * 90)
print()

# Compute statistics
stats = {}
for m in METHOD_NAMES:
    fl = np.array(all_results[m]['final_losses'])
    mm = np.array(all_results[m]['matmuls'])
    stats[m] = {
        'mean_loss': np.mean(fl),
        'std_loss': np.std(fl),
        'median_loss': np.median(fl),
        'mean_matmuls': np.mean(mm),
        'losses': fl,
        'matmuls': mm,
    }

# Recovery %: (loss_SGD - loss_method) / (loss_SGD - loss_FullDemocratic) * 100
# Computed per-seed then averaged
recoveries = {m: [] for m in METHOD_NAMES}
for i in range(N_SEEDS):
    sgd_l = all_results['SGD']['final_losses'][i]
    dem_l = all_results['FullDemocratic']['final_losses'][i]
    gap = sgd_l - dem_l
    for m in METHOD_NAMES:
        ml = all_results[m]['final_losses'][i]
        if abs(gap) > 1e-12:
            rec = (sgd_l - ml) / gap * 100.0
        else:
            rec = 0.0
        recoveries[m].append(rec)

for m in METHOD_NAMES:
    stats[m]['mean_recovery'] = np.mean(recoveries[m])
    stats[m]['std_recovery'] = np.std(recoveries[m])
    stats[m]['recoveries'] = np.array(recoveries[m])

# Pareto score: Recovery% / (matmuls / matmuls_SGD)
# Higher is better: high recovery at low cost
for m in METHOD_NAMES:
    cost_ratio = stats[m]['mean_matmuls'] / stats['SGD']['mean_matmuls']
    stats[m]['cost_ratio'] = cost_ratio
    if cost_ratio > 0:
        stats[m]['pareto_score'] = stats[m]['mean_recovery'] / cost_ratio
    else:
        stats[m]['pareto_score'] = 0.0

print("Statistics computed: mean_loss, std_loss, recovery%, pareto_score for all methods.")
print()
print("Intermediate check -- per-seed final losses:")
for m in METHOD_NAMES:
    losses_str = ", ".join([f"{l:.6f}" for l in stats[m]['losses']])
    print(f"  {DISPLAY_NAMES[m]:<28}: [{losses_str}]")
print()
print("Intermediate check -- per-seed matmul counts:")
for m in METHOD_NAMES:
    mms_str = ", ".join([f"{int(mm)}" for mm in stats[m]['matmuls']])
    print(f"  {DISPLAY_NAMES[m]:<28}: [{mms_str}]")

## Main Results Table: Final Loss, Recovery, and Pareto Efficiency

This is the primary results table. Each row corresponds to one optimizer, with columns showing:

- **Final Loss:** Mean loss at step 500 across 5 seeds (lower is better)
- **Std:** Standard deviation across seeds (lower is more reliable)
- **MatMuls:** Total matrix multiplications (lower is cheaper)
- **Recovery%:** Fraction of the SGD-to-Full-Democratic gap captured (higher is better)
- **Pareto:** Recovery% / relative cost (higher is more efficient)

### What to Look For

1. **Does Muon 2.0 (Lanczos-10) approach Full Democratic?** If its recovery is >80%, the Lanczos sketch captures most of the Hessian's structure with only 10 vectors.
2. **Does Muon 2.0 beat standard Muon?** If yes, the Hessian eigenbasis provides information that the polar factor's SVD basis does not.
3. **Is Lanczos-5 sufficient, or does Lanczos-10 provide a meaningful improvement?** This reveals how many extreme eigenvalues need equalization.
4. **Which method is Pareto-optimal?** The method with the highest Pareto score offers the best quality-cost trade-off.

In [ ]:
# ── Print main results table ─────────────────────────────────────────
print(f"{'Method':<28} {'Final Loss':>12} {'+-Std':>10} {'Matmuls':>10} {'Recovery%':>12} {'Pareto':>10}")
print("-" * 90)
for m in METHOD_NAMES:
    s = stats[m]
    print(f"{DISPLAY_NAMES[m]:<28} {s['mean_loss']:>12.6f} {s['std_loss']:>10.6f} "
          f"{s['mean_matmuls']:>10.0f} {s['mean_recovery']:>11.1f}% {s['pareto_score']:>10.1f}")
print()

## Per-Seed Recovery Analysis: Robustness Check

The per-seed recovery table reveals whether each method's advantage is consistent across random initializations or driven by one or two favorable seeds. A method with high mean recovery but enormous cross-seed variance is less trustworthy than one with moderate but consistent recovery.

Key diagnostics:
- **Uniform recovery across seeds:** The method genuinely captures curvature structure regardless of target alignment
- **Bimodal recovery (some seeds high, some low):** The method may be sensitive to the alignment between the Lanczos starting vector and the dominant Hessian eigenvectors
- **Recovery > 100% on some seeds:** The LR sweep found a regime where the method actually outperforms Full Democratic (possible if the democratic direction's norm matching creates a suboptimal effective step size)

In [ ]:
# ── Per-seed recovery table ──────────────────────────────────────────
print("Per-seed recovery % (relative to Full Democratic):")
print(f"{'Method':<28}", end="")
for i in range(N_SEEDS):
    print(f" {'Seed'+str(i):>8}", end="")
print(f" {'Mean':>8}")
print("-" * (28 + 9 * (N_SEEDS + 1)))
for m in METHOD_NAMES:
    print(f"{DISPLAY_NAMES[m]:<28}", end="")
    for i in range(N_SEEDS):
        print(f" {recoveries[m][i]:>7.1f}%", end="")
    print(f" {stats[m]['mean_recovery']:>7.1f}%")
print()

# Additional robustness statistics
print("Recovery variance analysis:")
for m in METHOD_NAMES:
    rec = np.array(recoveries[m])
    print(f"  {DISPLAY_NAMES[m]:<28}: mean={np.mean(rec):7.1f}%, std={np.std(rec):6.1f}%, "
          f"min={np.min(rec):7.1f}%, max={np.max(rec):7.1f}%, range={np.max(rec)-np.min(rec):6.1f}pp")

## Loss Trajectory: Seed 0 Convergence Dynamics

The loss trajectory at seed 0 provides a temporal view of optimization dynamics. Key features to watch for:

- **Initial convergence rate:** Which methods make the fastest progress in the first 50-100 steps? Methods with better initial directions can exploit the strong-curvature phase where the loss surface is most amenable to large steps.
- **Plateau behavior:** Does any method hit a plateau where the loss stops decreasing? This often indicates that the optimizer is stuck in a low-curvature direction and needs better spectral information.
- **Late-stage convergence:** As the loss approaches the minimum, do the Hessian-aware methods maintain their advantage, or does the curvature landscape become more isotropic (reducing the benefit of equalization)?
- **Convergence order changes:** If one method leads early but another catches up later, it suggests different strengths at different phases of training.

In [ ]:
# ── Loss trajectory (seed 0) ─────────────────────────────────────────
print("Loss trajectory (seed 0, every 50 steps):")
print(f"{'Step':>5}", end="")
for m in METHOD_NAMES:
    print(f" {DISPLAY_NAMES[m][:14]:>14}", end="")
print()
for s in list(range(0, N_STEPS, 50)) + [N_STEPS - 1]:
    print(f"{s:>5}", end="")
    for m in METHOD_NAMES:
        lc = all_results[m]['loss_curves'][0]
        if s < len(lc):
            print(f" {lc[s]:>14.6f}", end="")
        else:
            print(f" {'N/A':>14}", end="")
    print()
print()

# Additional trajectory analysis: log-loss for convergence rate estimation
print("Log-loss at key steps (seed 0) -- for estimating convergence rates:")
print(f"{'Step':>5}", end="")
for m in METHOD_NAMES:
    print(f" {m[:10]:>12}", end="")
print()
for s in [0, 50, 100, 200, 300, 499]:
    print(f"{s:>5}", end="")
    for m in METHOD_NAMES:
        lc = all_results[m]['loss_curves'][0]
        if s < len(lc) and lc[s] > 0:
            print(f" {np.log10(lc[s]):>12.3f}", end="")
        else:
            print(f" {'N/A':>12}", end="")
    print()
print()
print("Interpretation:")
print("  Equal spacing in log-loss indicates exponential convergence (constant rate).")
print("  Narrowing gaps indicate convergence slowdown (curvature-limited phase).")
print("  Methods that maintain wide gaps with SGD in late steps have durable advantages.")

## Computational Cost Analysis: MatMul Breakdown

This section decomposes the computational cost of each optimizer into its constituent operations. The matmul count is used as a hardware-independent proxy for floating-point cost.

### Cost Components

- **SGD:** Each step requires one forward pass ($L-1 = 2$ matmuls) and one gradient computation (~$3L$ matmuls for the 3-term backprop chain). Total: $\sim 11$ matmuls/step.
- **Muon:** Same gradient cost, plus 5 Newton-Schulz iterations per layer, each involving 2 matmuls ($X^T X$ and $X \cdot (X^T X)$). Total overhead: $5 \times 2 \times 3 = 30$ matmuls/step.
- **Lanczos variants:** 10 (or 5) HVP evaluations per recomputation, each requiring 2 gradient evaluations. Amortized over 50 steps. The per-step cost is modest but includes an overhead spike every 50 steps.
- **Hessian rescale:** 4 HVPs per layer per step (one per singular direction), costing $4 \times 3 = 12$ gradient-equivalent evaluations per step.

### Why Cost Matters

A method that achieves 110% of Muon's recovery at 10x the cost is useless in practice. The Pareto analysis identifies the *efficient frontier*: methods that achieve the highest recovery at each cost level. Only methods on this frontier are worth considering for practical deployment.

In [ ]:
# ── Matmul cost breakdown ────────────────────────────────────────────
print("Matmul cost analysis:")
print(f"{'Method':<28} {'Total matmuls':>14} {'Cost ratio':>12} {'Notes'}")
print("-" * 80)
for m in METHOD_NAMES:
    s = stats[m]
    notes = ""
    if m == 'SGD':
        notes = "baseline (1 grad/step)"
    elif m == 'Muon':
        notes = "1 grad + k=5 NS iters/step"
    elif m == 'FullDemocratic':
        notes = "1 grad + full Hessian every 50 steps (ORACLE)"
    elif m == 'Muon2_L10':
        notes = "1 grad + 10 Lanczos steps every 50 steps"
    elif m == 'Muon2_L5':
        notes = "1 grad + 5 Lanczos steps every 50 steps"
    elif m == 'Muon+Hrescale':
        notes = "Muon + 4 HVPs/layer/step"
    print(f"{DISPLAY_NAMES[m]:<28} {s['mean_matmuls']:>14.0f} {s['cost_ratio']:>11.2f}x  {notes}")
print()

# Additional cost analysis
print("Cost interpretation:")
sgd_cost = stats['SGD']['mean_matmuls']
print(f"  SGD baseline: {sgd_cost:.0f} matmuls total = {sgd_cost/N_STEPS:.1f} matmuls/step")
for m in METHOD_NAMES:
    if m == 'SGD':
        continue
    s = stats[m]
    overhead = s['mean_matmuls'] - sgd_cost
    overhead_pct = overhead / sgd_cost * 100
    per_step_overhead = overhead / N_STEPS
    print(f"  {DISPLAY_NAMES[m]:<28}: +{overhead:.0f} matmuls overhead (+{overhead_pct:.1f}%), "
          f"+{per_step_overhead:.1f} matmuls/step")

## Hypothesis Tests: Four Core Questions

Each test addresses a specific prediction about the relationship between Hessian-aware equalization and optimization performance.

### T1: Can a Lanczos Sketch Capture >80% of Full Democratic?

This tests whether 10 Ritz vectors (out of 48 total dimensions) are sufficient to capture the essential curvature structure. The 80% threshold is chosen because:
- Below 80%, the sketch is missing too much structure to be a viable replacement
- Above 80%, the remaining gap may be due to subspace complement noise rather than missing essential directions

### T2: Does Lanczos-10 Beat Standard Muon?

This is the most important test. If Muon 2.0 (Lanczos-10) beats standard Muon, it proves that:
1. The Hessian eigenbasis contains information that the gradient's SVD basis does not
2. This information can be extracted cheaply enough via Lanczos to be practical
3. There is room for improvement beyond the polar factor

### T3: Is Lanczos-5 Sufficient?

If even 5 Ritz vectors beat standard Muon, the curvature structure is dominated by a small number of extreme eigenvalues. This would suggest that the ill-conditioning problem is low-rank in nature -- only a few directions are severely misscaled.

### T4: Does Curvature-Aware Magnitude Help?

Muon + Hessian rescaling tests whether the polar factor's uniform-magnitude property is optimal, or whether adapting the magnitude to local curvature provides additional benefit. A positive result would suggest that Muon's direction is good but its magnitude is suboptimal.

In [ ]:
# ── Key hypothesis tests ─────────────────────────────────────────────
print("=" * 90)
print("KEY HYPOTHESIS TESTS")
print("=" * 90)
print()

# T1: Muon 2.0 (Lanczos-10) recovery > 80%?
rec_l10 = stats['Muon2_L10']['mean_recovery']
t1_pass = rec_l10 > 80.0
print(f"T1: Muon 2.0 (Lanczos-10) recovers >80% of Full Democratic advantage?")
print(f"    Mean recovery = {rec_l10:.1f}%  (per-seed: {stats['Muon2_L10']['recoveries'].round(1)})")
print(f"    --> {'PASS' if t1_pass else 'FAIL'}")
print()

# T2: Muon 2.0 (Lanczos-10) beats standard Muon?
rec_muon = stats['Muon']['mean_recovery']
t2_pass = rec_l10 > rec_muon
print(f"T2: Muon 2.0 (Lanczos-10) beats standard Muon?")
print(f"    Muon 2.0 recovery = {rec_l10:.1f}%  vs  Muon recovery = {rec_muon:.1f}%")
print(f"    Gap: {rec_l10 - rec_muon:+.1f} percentage points")
print(f"    --> {'PASS' if t2_pass else 'FAIL'}")
print()

# T3: Muon 2.0 (Lanczos-5) -- does even the cheap version help?
rec_l5 = stats['Muon2_L5']['mean_recovery']
t3_pass = rec_l5 > rec_muon
print(f"T3: Muon 2.0 (Lanczos-5) beats standard Muon?")
print(f"    Muon 2.0 (L5) recovery = {rec_l5:.1f}%  vs  Muon recovery = {rec_muon:.1f}%")
print(f"    Gap: {rec_l5 - rec_muon:+.1f} percentage points")
print(f"    --> {'PASS' if t3_pass else 'FAIL'}")
print()

# T4: Muon + Hessian rescaling beats standard Muon?
rec_hrescale = stats['Muon+Hrescale']['mean_recovery']
t4_pass = rec_hrescale > rec_muon
print(f"T4: Muon + Hessian rescaling beats standard Muon?")
print(f"    Muon+Hrescale recovery = {rec_hrescale:.1f}%  vs  Muon recovery = {rec_muon:.1f}%")
print(f"    Gap: {rec_hrescale - rec_muon:+.1f} percentage points")
print(f"    --> {'PASS' if t4_pass else 'FAIL'}")
print()

## Pareto Efficiency Analysis: Best Quality-Cost Trade-off

The Pareto analysis answers the practical question: **if you have a fixed computational budget, which method gives you the most optimization benefit?**

A method is on the **Pareto frontier** if no other method simultaneously achieves higher recovery AND lower cost. The Pareto score (recovery / relative cost) provides a simple scalar summary, but the full picture requires examining the recovery-vs-cost scatter:

- A method in the upper-left (high recovery, low cost) dominates
- A method in the lower-right (low recovery, high cost) is dominated
- Methods along the diagonal represent different points on the efficiency frontier

In [ ]:
# T5: Best Pareto-efficient method?
best_pareto = max(METHOD_NAMES, key=lambda m: stats[m]['pareto_score'])
print(f"T5: Best Pareto score (recovery / relative cost)?")
for m in METHOD_NAMES:
    marker = " <-- BEST" if m == best_pareto else ""
    print(f"    {DISPLAY_NAMES[m]:<28} Pareto = {stats[m]['pareto_score']:.1f}  "
          f"(Recovery={stats[m]['mean_recovery']:.1f}%, Cost={stats[m]['cost_ratio']:.2f}x){marker}")
print()

# Pareto frontier identification
print("Pareto frontier analysis:")
sorted_methods = sorted(METHOD_NAMES, key=lambda m: stats[m]['cost_ratio'])
pareto_frontier = []
best_recovery_so_far = -1e10
for m in sorted_methods:
    if stats[m]['mean_recovery'] > best_recovery_so_far:
        pareto_frontier.append(m)
        best_recovery_so_far = stats[m]['mean_recovery']
print(f"  Methods on the Pareto frontier (in order of increasing cost):")
for m in pareto_frontier:
    print(f"    {DISPLAY_NAMES[m]:<28}: Recovery={stats[m]['mean_recovery']:.1f}%, Cost={stats[m]['cost_ratio']:.2f}x")
non_pareto = [m for m in METHOD_NAMES if m not in pareto_frontier]
if non_pareto:
    print(f"  Dominated methods (not on frontier):")
    for m in non_pareto:
        print(f"    {DISPLAY_NAMES[m]:<28}: Recovery={stats[m]['mean_recovery']:.1f}%, Cost={stats[m]['cost_ratio']:.2f}x")

## Summary Verdict: Does the Hessian Sketch Approach Work?

The verdict synthesizes all four hypothesis tests and the Pareto analysis into a single assessment.

### Possible Outcomes and Their Implications

**Strong Result (T1 and T2 pass):** Muon 2.0 (Lanczos-10) recovers >80% of Full Democratic AND beats standard Muon. This proves:
- The gap between Muon and Full Democratic is real and exploitable
- A cheap Lanczos sketch captures sufficient Hessian structure
- There is actionable room for improvement beyond the polar factor

**Positive Result (T2 passes, T1 fails):** Lanczos-10 beats Muon but does not reach 80%. The Hessian sketch helps, but more Ritz vectors or more frequent updates may be needed.

**Mixed Result (T1 passes, T2 fails):** High recovery but Muon is already near the optimum. The polar factor may already be capturing most of the Hessian structure at this problem size.

**Negative Result (both fail):** The Lanczos approach does not help. Possible reasons:
1. 10 Ritz vectors are too few to span the important curvature directions
2. The 50-step recomputation interval causes excessive staleness
3. The polar factor is already near-optimal, and the remaining gap is not addressable via partial equalization

### Connection to Experiment 2.11

Experiment 2.11 established that Full Democratic SGD achieves ~150% of Muon's advantage over SGD. If Muon 2.0 (Lanczos-10) recovers $R$% of Full Democratic, then it achieves $R/100 \times 150\% = 1.5R\%$ of Muon's advantage. Standard Muon by definition achieves $\text{rec}_{\text{Muon}}/100 \times 150\%$ of its own advantage.

In [ ]:
# ── Summary verdict ──────────────────────────────────────────────────
print("=" * 90)
print("SUMMARY VERDICT")
print("=" * 90)
print()

n_pass = sum([t1_pass, t2_pass, t3_pass, t4_pass])
print(f"Tests passed: {n_pass}/4")
print()

if t1_pass and t2_pass:
    print("STRONG RESULT: Muon 2.0 (Lanczos-10) recovers >80% of the full democratic")
    print("advantage AND beats standard Muon. The Hessian sketch approach works.")
elif t2_pass:
    print("POSITIVE RESULT: Muon 2.0 (Lanczos-10) beats standard Muon, though it does")
    print("not reach 80% of full democratic advantage. The Lanczos sketch helps but")
    print("more eigenvectors or more frequent updates may be needed.")
elif t1_pass:
    print("MIXED: High recovery but doesn't beat Muon -- standard Muon may already be")
    print("near-optimal for this problem size.")
else:
    print("NEGATIVE: The Lanczos sketch approach did not significantly improve over Muon")
    print("in this setting. Possible reasons: too few eigenvectors, stale Hessian info,")
    print("or the polar factor is already capturing the essential structure.")

print()
if t4_pass:
    print("BONUS: Muon + Hessian rescaling improves over vanilla Muon, suggesting that")
    print("curvature-aware step sizing complements the polar factor's direction choice.")
else:
    print("NOTE: Muon + Hessian rescaling did NOT improve over vanilla Muon. The polar")
    print("factor may already provide sufficient curvature adaptation, or the rescaling")
    print("scheme needs refinement.")

print()
print(f"Best Pareto method: {DISPLAY_NAMES[best_pareto]} (score={stats[best_pareto]['pareto_score']:.1f})")
print()

# Final comparison with 2.11
print("Comparison with Experiment 2.11 results:")
print(f"  2.11 showed Full Democratic SGD at ~150% of Muon's advantage")
print(f"  Here: Muon 2.0 (Lanczos-10) at {rec_l10:.0f}% of Full Democratic = "
      f"{rec_l10 * 1.0:.0f}% of Full Democratic, which is {rec_l10 / 100.0 * 150.0:.0f}% of Muon's advantage")
print(f"  Standard Muon at {rec_muon:.0f}% of Full Democratic (={rec_muon / 100.0 * 150.0:.0f}% of Muon's advantage by 2.11 metric)")
print()
print("=" * 90)

## Conclusions and Interpretation

### What This Experiment Establishes

This experiment directly tests the hierarchy of optimization directions by comparing methods that equalize the gradient in progressively richer spectral bases:

1. **No equalization** (SGD): Uses the raw gradient, which is dominated by high-curvature directions
2. **Per-layer SVD equalization** (Muon): Equalizes within each layer's gradient matrix, but misses inter-layer Hessian structure
3. **Partial Hessian equalization** (Muon 2.0): Equalizes in the extreme eigenspaces of the global Hessian, capturing the worst anisotropies
4. **Full Hessian equalization** (Full Democratic): The oracle upper bound on what any equalization scheme can achieve

The key question is whether positions 2 and 3 can be meaningfully separated, and at what computational cost.

### Relationship to the Muon-as-RG-Gauge-Fixing Theory

In the gauge-fixing interpretation of Muon:
- The polar factor fixes the **per-layer orthogonal gauge**, removing redundant degrees of freedom within each layer
- The Hessian eigenbasis represents the **global curvature structure** of the loss landscape, which includes inter-layer coupling
- The gap between Muon and Full Democratic represents the cost of gauge-fixing at the layer level rather than globally

If Muon 2.0 closes this gap, it suggests that a small number of global Hessian eigenvectors capture the inter-layer coupling that per-layer gauge-fixing misses. This would be a significant refinement: the full Hessian has $O(n^2)$ entries, but the effective dimensionality of the inter-layer coupling may be only $O(k)$ with $k \ll n$.

### Practical Implications

- **If Muon 2.0 works:** It suggests a practical path to a better optimizer -- run a few Lanczos iterations periodically to sketch the Hessian's extreme structure, and use this to augment Muon's per-layer polar factor. The overhead is modest (a few extra gradient evaluations every $N$ steps).
- **If Muon 2.0 fails:** The polar factor may already be a near-optimal practical approximation to the democratic ideal, at least for the problem sizes and condition numbers tested here. Improvements would need to come from other axes (e.g., momentum, schedule, architecture).

### Caveats

1. **Tiny problem size:** $n = 48$ parameters is extremely small. At larger scales, the Lanczos approach becomes relatively cheaper (amortized cost decreases) but the Hessian structure may be more complex.
2. **Deep linear network:** The Hessian structure of deep linear networks is well-understood and may be unusually amenable (or resistant) to low-rank spectral sketching.
3. **No momentum:** All methods use vanilla gradient descent without momentum. Adding momentum could change the relative rankings.
4. **Fixed recomputation interval:** The 50-step interval was not optimized. Adaptive recomputation (e.g., when the Ritz vectors' explanatory power drops below a threshold) could improve results.